# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [1]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

import numpy as np
np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

## Parameters

In [2]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2025, 1, 1)
END_DATE   = datetime.date(2026, 4, 5)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-4%-per-shard wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 2, 15)

COPY_WINDOW_SECONDS = 5 * 60  # 5 minutes

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
ENRICHED_TRADES_DIR  = Path("../../data/trades_polygon_enriched")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


In [7]:
tdf = pd.read_parquet(ENRICHED_TRADES_DIR / "enriched_0.parquet")
len(tdf)

7033417

## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

We parse every `market_json` to build:
1. **token_lookup** – maps `token_id → {condition_id, outcome, token_winner, final_price}`
2. **market_meta** – per-market DataFrame with `condition_id`, `question`, `end_date`, `market_slug`

In [3]:
# Load only the market files whose month falls within [START_DATE, END_DATE].
import datetime as _dt
def _file_in_range(p, start, end):
    """Return True if YYYY-MM.parquet filename falls within the date range."""
    try:
        y, m = (int(x) for x in p.stem.split("-"))
        file_date = _dt.date(y, m, 1)
        return start.replace(day=1) <= file_date <= end.replace(day=1)
    except Exception:
        return False

market_files = [
    p for p in sorted(MARKETS_DIR.glob("*.parquet"))
    if _file_in_range(p, START_DATE, END_DATE)
]
print(f"Found {len(market_files)} market file(s)")

all_market_rows = pd.concat(
    [pd.read_parquet(f) for f in market_files],
    ignore_index=True,
)
# deduplicate by condition_id (keep first occurrence)
all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
print(f"Unique markets (all):  {len(all_market_rows):,}")

# ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# Parse end_date_iso as a date; keep only markets whose resolution date falls
# within [START_DATE, END_DATE] (inclusive).  Markets outside this range are
# excluded, and their tokens will not appear in the token_lookup, so any trades
# on those tokens will be dropped during the enrichment join.
end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
all_market_rows = all_market_rows[in_range].reset_index(drop=True)
print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
all_market_rows.head(2)

Found 16 market file(s)
Unique markets (all):  692,708
Unique markets (filtered 2025-01-01 → 2026-04-05): 665,930


,end_date_iso,condition_id,market_json
0,2025-01-01T00:00:00Z,,"{""enable_order_book"":false,""active"":false,""clo..."
1,2025-01-01T00:00:00Z,0x0aff77a831ea1fb2cb57d55c87f3f12fdf734cac6471...,"{""enable_order_book"":false,""active"":true,""clos..."


In [4]:
def build_lookups(market_rows: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Parse market_json column and return (token_lookup, market_meta_df)."""
    token_lookup: dict[str, dict] = {}
    meta_rows: list[dict] = []

    skipped_markets = 0

    for _, row in market_rows.iterrows():
        try:
            m = json_loads(row["market_json"])
        except Exception:
            skipped_markets += 1
            continue

        cid = str(m.get("condition_id", row["condition_id"]))

        has_winner = False

        for tok in m.get("tokens", []):
            w = bool(tok.get("winner", None))
            if(w):
                has_winner = True
                break

        if not has_winner:
            skipped_markets += 1
            continue


        # ── token resolution lookup ───────────────────────────────────────────
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", None))

            # final_price is determined solely by the token winner flag:
            # winner=True  → settled at 1.0 USDC per share
            # winner=False → settled at 0.0 USDC per share
            final_price = 1.0 if winner else 0.0
            token_lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }

        # ── market metadata ───────────────────────────────────────────────────
        meta_rows.append({
            "condition_id": cid,
            "question":     m.get("question", ""),
            "end_date":     pd.to_datetime(m.get("end_date_iso"), utc=True, errors="coerce"),
            "market_slug":  m.get("market_slug", ""),
        })
    
    print(f"Skipped markets due to JSON parsing or missing winner flag: {skipped_markets}")

    market_meta = pd.DataFrame(meta_rows)
    return token_lookup, market_meta


token_lookup, market_meta = build_lookups(all_market_rows)
print(f"Token lookup entries: {len(token_lookup):,}")
print(f"Market meta rows:     {len(market_meta):,}")

Skipped markets due to JSON parsing or missing winner flag: 29690
Token lookup entries: 1,272,480
Market meta rows:     636,240


## 2 · Process trades — Phase 1: select top-4% wallets per shard

Each shard is processed independently in parallel.  Only per-wallet training-period P&L
is returned (no trade rows), so memory usage is minimal.  The union of top-4% wallets
across all shards becomes the candidate set for Phase 2.

In [5]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

from polymarket_analysis.preprocessing.shard_processor import (
    select_top_wallets_shard,
    enrich_and_group_shard,
)

trade_files = sorted(TRADES_DIR.glob("*.parquet"))
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

print(f"Found {len(trade_files)} trade shard(s)")

sample_raw = pd.read_parquet(trade_files[0])
print(f"Sample shard rows ({trade_files[0].name}): {len(sample_raw):,}")
print(f"Columns: {sample_raw.columns.tolist()}")
if "trade_date" in sample_raw.columns:
    print(f"Date range: {sample_raw['trade_date'].min()} → {sample_raw['trade_date'].max()}")
sample_raw.head(3)

Found 16 trade shard(s)
Sample shard rows (0.parquet): 7,033,417
Columns: ['tx_hash', 'log_index', 'block_number', 'block_timestamp', 'trade_date', 'condition_id', 'token_id', 'outcome', 'token_winner', 'wallet', 'side', 'price', 'quantity', 'usdc_amount', 'question', 'slug', 'fill_count', 'position', 'flipped', 'split', 'tags', 'raw_side', 'raw_outcome', 'raw_token_id', 'raw_price', 'raw_usdc_amount']
Date range: 2025-01-01 → 2026-03-28


,tx_hash,log_index,block_number,block_timestamp,trade_date,condition_id,token_id,outcome,token_winner,wallet,...,fill_count,position,flipped,split,tags,raw_side,raw_outcome,raw_token_id,raw_price,raw_usdc_amount
0,0xf950fe06f8409f05d5ad6a5029df3ea2156c20e45737...,413,66158926,1735689617,2025-01-01,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,2760237849528716269099862867806998297850735825...,Lakers,False,0x4a64afa45a44a01890c2161be88d2b44751d4430,...,1,122.04,False,False,"[Sports, NBA, Games]",BUY,Lakers,2760237849528716269099862867806998297850735825...,0.32,39.05
1,0xf950fe06f8409f05d5ad6a5029df3ea2156c20e45737...,413,66158926,1735689617,2025-01-01,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,4095939065153142593574376695211848274631622064...,Cavaliers,True,0xecb164b37ae2f584fb56e23b7f72719920b14ae4,...,1,122.04,True,False,"[Sports, NBA, Games]",SELL,Lakers,2760237849528716269099862867806998297850735825...,0.32,39.05
2,0xf950fe06f8409f05d5ad6a5029df3ea2156c20e45737...,421,66158926,1735689617,2025-01-01,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,2760237849528716269099862867806998297850735825...,Lakers,False,0x54f59ff063411a19d244ed2a345ad94e79cde936,...,1,172.08,False,False,"[Sports, NBA, Games]",BUY,Lakers,2760237849528716269099862867806998297850735825...,0.32,55.06


In [6]:
# Preprocess: enrich by copyable quantity if not present

from concurrent.futures import ProcessPoolExecutor

from polymarket_analysis.preprocessing.fill_extender import enrich_shard

with ProcessPoolExecutor() as executor:
    futures = {executor.submit(enrich_shard, f, ENRICHED_TRADES_DIR, COPY_WINDOW_SECONDS): f for f in trade_files}

    total = len(futures)

    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"Error processing {f}: {e}")
            continue

        if i % 4 == 0 or i == total:
            print(f"{i}/{total} shards processed")

Enriched file for 0.parquet already exists, skipping...
Enriched file for 4.parquet already exists, skipping...
Enriched file for 5.parquet already exists, skipping...
Enriched file for 6.parquet already exists, skipping...
Enriched file for 8.parquet already exists, skipping...
Enriched file for 9.parquet already exists, skipping...
Enriched file for 1.parquet already exists, skipping...
Enriched file for a.parquet already exists, skipping...
Enriched file for b.parquet already exists, skipping...
Enriched file for 2.parquet already exists, skipping...
4/16 shards processed
8/16 shards processed
Enriched file for e.parquet already exists, skipping...
Enriched file for d.parquet already exists, skipping...
Enriched file for f.parquet already exists, skipping...
12/16 shards processed
Enriched file for 3.parquet already exists, skipping...
Enriched file for 7.parquet already exists, skipping...
Enriched file for c.parquet already exists, skipping...
16/16 shards processed


In [7]:
# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', 200)

# MARKET = '0x02d03a859b9d64c7cbbc2a2c3172898bb89b379dba10d54d74ee2c1d5036a71b'

# df = pd.read_parquet(ENRICHED_TRADES_DIR / "enriched_0.parquet")
# df = df[df['condition_id'] == MARKET]
# df[['side', 'price', 'quantity', 'ts', 'token_id', 'tx_hash', 'wallet', 'position']].head(5)

In [8]:
# len(df)
# df[df['tx_hash']=='0xffece5c5cde2b0e1b9375cf30cbb3af09f087967143aff3b4fe5e53c8d1b58c3']

In [9]:
# df['ts'] = pd.to_datetime(df['block_timestamp'], unit='s')
# df['token_id'] = df['token_id'].str[:5]
# df['tx_hash'] = df['tx_hash'].str[:5]
# df[["wallet", 'tx_hash', 'ts', "side", "price", "quantity","token_id", "token_winner", "avail_copy_qty", "avail_copy_total_vol", "avail_copy_count","condition_id"]].head(1)

In [10]:
# ── build token resolution DataFrame ────────────────────────────────────────
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "token_id"
token_df.reset_index(inplace=True)
token_df["token_id"] = token_df["token_id"].astype(str)

END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

enriched_trade_files = sorted(ENRICHED_TRADES_DIR.glob("*.parquet"))

# ── Phase 1: identify top-4% wallets per shard (parallel) ────────────────────
print("Phase 1 — selecting top-4% wallets per shard...")
shard_wallet_pnls: dict[str, float] = {}   # wallet → best shard pnl (for diagnostics)
shard_wallet_thresholds: dict[int, float] = {}   # shard index → top-pct threshold (for diagnostics)
total_raw_rows      = 0
total_in_range_rows = 0
total_candidates    = 0

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(select_top_wallets_shard, f, token_df, END_TRAIN_TS, top_pct=0.04): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        wallet_pnl, stats = fut.result()
        total_raw_rows      += stats["raw_rows"]
        total_in_range_rows += stats["in_range_rows"]
        total_candidates    += stats["candidate_wallets"]
        # union: keep the wallet; if it appears in multiple shards take max pnl
        for w, pnl in wallet_pnl.items():
            if w not in shard_wallet_pnls or pnl > shard_wallet_pnls[w]:
                shard_wallet_pnls[w] = pnl
        shard_wallet_thresholds[i] = stats["threshold"]
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(
                f"  {i}/{len(enriched_trade_files)} shards | "
                f"raw: {total_raw_rows:,} | in-range: {total_in_range_rows:,} | "
                f"wallet union so far: {len(shard_wallet_pnls):,}"
            )

top_wallets: set[str] = set(shard_wallet_pnls.keys())
print(f"\nPhase 1 done. Candidate wallets (union of top-4% per shard): {len(top_wallets):,}")
print(f"Threshold pnls per shard: {shard_wallet_thresholds}")

Phase 1 — selecting top-4% wallets per shard...
Processing shard enriched_0.parquet...
Processing shard enriched_1.parquet...
Processing shard enriched_2.parquet...
Processing shard enriched_3.parquet...
Processing shard enriched_4.parquet...
Processing shard enriched_5.parquet...
Processing shard enriched_6.parquet...
Shard top 4.00% threshold: 74.67 USDC
Processing shard enriched_7.parquet...
Shard top 4.00% threshold: 87.10 USDC
Processing shard enriched_8.parquet...
Processing shard enriched_9.parquet...
Processing shard enriched_a.parquet...
Shard top 4.00% threshold: 94.22 USDC
Processing shard enriched_b.parquet...
Shard top 4.00% threshold: 95.94 USDC
Processing shard enriched_c.parquet...
Shard top 4.00% threshold: 84.52 USDC
  4/16 shards | raw: 28,458,061 | in-range: 28,239,780 | wallet union so far: 34,127
Processing shard enriched_d.parquet...
Shard top 4.00% threshold: 93.49 USDC
Processing shard enriched_e.parquet...
Shard top 4.00% threshold: 81.74 USDC
Shard top 4.00% 

## 3 · Phase 2: enrich, group by tx×wallet×side, filter to top wallets

Each shard is processed in parallel.  Fills are inner-joined with settlement data,
aggregated to one row per ``tx_hash × wallet × side``, and filtered to the wallet set
from Phase 1.  Results are concatenated in memory and re-grouped to merge any
transactions that span shard boundaries.

In [11]:
# ── Phase 2: enrich + group + filter (parallel) ──────────────────────────────
print("Phase 2 — grouping and filtering shards...")
shard_results:     list[pd.DataFrame]    = []
wallet_pnl_total:  dict[str, float]      = {}   # accumulated training P&L per wallet
sample_fills = None

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(enrich_and_group_shard, f, token_df, END_TRAIN_TS, top_wallets): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        grouped_shard, shard_train_pnl = fut.result()
        if not grouped_shard.empty:
            shard_results.append(grouped_shard)
            if sample_fills is None:
                sample_fills = grouped_shard.head(5).copy()
        for w, pnl in shard_train_pnl.items():
            wallet_pnl_total[w] = wallet_pnl_total.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(f"  {i}/{len(enriched_trade_files)} shards | shards with data: {len(shard_results)}")

if not shard_results:
    raise ValueError("No in-range trade rows after token filter")

# ── merge cross-shard partial groups ─────────────────────────────────────────
GROUP_KEYS = ["tx_hash", "wallet", "side"]
grouped = pd.concat(shard_results, ignore_index=True)
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum",  "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
        copyable_pnl     = ("copyable_pnl",     "sum"),
        copyable_qty    = ("copyable_qty",   "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)
grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
# grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "dt"], inplace=True, ignore_index=True)

print(f"\nPhase 2 done.")
print(f"  Grouped rows (all top wallets, all periods): {len(grouped):,}")
print(f"  Unique wallets in grouped:                   {grouped['wallet'].nunique():,}")
grouped.head(5)

Phase 2 — grouping and filtering shards...
  4/16 shards | shards with data: 4
  8/16 shards | shards with data: 8
  12/16 shards | shards with data: 12
  16/16 shards | shards with data: 16

Phase 2 done.
  Grouped rows (all top wallets, all periods): 57,737,372
  Unique wallets in grouped:                   80,271


,tx_hash,wallet,side,dt,condition_id,outcome,token_winner,final_price,position,total_quantity,price_x_qty_sum,trade_value_usdc,final_value_usdc,num_fills,trade_pnl,copyable_pnl,copyable_qty,avail_copy_total_vol,avail_copy_count,avg_price
0,0x7c0fecf5fb55d26ae8e855c2c1910832ed0aa552ca69...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-22 12:31:11+00:00,0x9c4b9abda44275da5a1f1b2822dd0d49f4adcfffb731...,Clippers,True,1.00,683.45,683.45,362.23,362.23,683.45,1,321.22,321.22,683.45,1100.02,4.00,0.53
1,0x8e3ac2eeb09a1db32f2839685a174c97cc28ea05f18a...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-23 08:50:15+00:00,0xaaca3c792f030c897d98ea2f8a824d1842a3f31d054b...,Heat,True,1.00,1366.90,1366.90,683.45,683.45,1366.90,1,683.45,395.61,791.22,403.52,6.00,0.50
2,0xff336f7d5ee78018842d2ed0654df802cf40d725c8a1...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-24 12:45:17+00:00,0x1c1ece687825a5be15c65896cd0ee0f8cf06e1bc2a87...,Cavaliers,False,0.00,922.04,922.04,497.90,497.90,0.00,3,-497.90,-466.15,863.25,3792.33,12.00,0.54
3,0x16d07796a1425eed307a85ea9d64411808115b84aa62...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-26 11:52:43+00:00,0x42b616c90a02e4e01bfed0aa673a1320f966e3381bef...,Hornets,False,0.00,1700.00,1700.00,510.00,510.00,0.00,8,-510.00,-20.01,66.69,21.04,16.00,0.30
4,0x468032632d1a852711e7e1232be55e2f0ce1ef885751...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-27 12:58:45+00:00,0xe77c389d59082205bc4f8881a21902aaecbde16521f6...,Packers,True,1.00,872.39,872.39,357.68,357.68,872.39,1,514.71,514.71,872.39,5188.59,14.00,0.41


## 4 · Phase 3: apply final PnL cut-off and export

Compute the true cross-shard training P&L per wallet.  Use the **median** of the
per-shard P&L values accumulated in Phase 1 as the export cut-off: wallets whose
total training P&L falls below that median are dropped before writing.

In [12]:
# ── wallet summary (full cross-shard training P&L) ───────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < END_TRAIN_TS]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets     = ("condition_id",    "nunique"),
        num_trades      = ("num_fills",        "sum"),
        total_cost_usdc = ("trade_value_usdc", "sum"),
        copyable_pnl       = ("copyable_pnl",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
    )
    .sort_values("copyable_pnl", ascending=False)
    .reset_index()
)

# ── PnL cut-off: median of the per-shard pnl values collected in Phase 1 ─────
import statistics
pnl_cutoff = statistics.median(shard_wallet_thresholds.values())

print(f"\nUnique wallets in training data: {wallet_summary['wallet'].nunique():,}")
print(f"wallet_summary['copyable_pnl'] quantiles:\n{wallet_summary['copyable_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}")

final_wallets = set(
    wallet_summary[
        (wallet_summary["copyable_pnl"] >= pnl_cutoff) &
        (wallet_summary["trade_pnl"] >= wallet_summary["copyable_pnl"])
    ]["wallet"]
)

print(f"\nFinal selected wallets (Phase 2 filter): {len(final_wallets):,}")
print(f"final_wallets['copyable_pnl'] quantiles:\n{wallet_summary[wallet_summary['wallet'].isin(final_wallets)]['copyable_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}")

print(f"Unique wallets after Phase 2 union:    {len(wallet_summary):,}")
print(f"Per-shard PnL median (cut-off):        {pnl_cutoff:,.2f} USDC")
print(f"Wallets passing PnL cut-off:           {len(final_wallets):,}")
wallet_summary.head(5)


Unique wallets in training data: 80,271
wallet_summary['copyable_pnl'] quantiles:
0.01   -22386.17
0.10    -1347.44
0.20     -426.27
0.30     -122.05
0.40       37.80
0.50      112.70
0.60      176.99
0.70      285.91
0.80      496.17
0.90     1057.50
0.99    11588.30
Name: copyable_pnl, dtype: float64

Final selected wallets (Phase 2 filter): 24,500
final_wallets['copyable_pnl'] quantiles:
0.01      89.18
0.10     114.44
0.20     146.51
0.30     188.96
0.40     246.11
0.50     327.67
0.60     451.61
0.70     639.92
0.80    1021.10
0.90    2162.00
0.99   28878.51
Name: copyable_pnl, dtype: float64
Unique wallets after Phase 2 union:    80,271
Per-shard PnL median (cut-off):        86.02 USDC
Wallets passing PnL cut-off:           24,500


,wallet,num_markets,num_trades,total_cost_usdc,copyable_pnl,trade_pnl
0,0xdb27bf2ac5d428a9c63dbc914611036855a6c56e,605,67274,38307581.47,1290124.14,4923915.29
1,0xe20a1538293903b746ffe6c4ce2d5c3c0300e469,4116,55427,36564774.56,933389.80,557979.44
2,0x07b8e44b90cc3e91b8d5fe60ea810d2534638e25,182,20476,14395104.80,723949.17,521486.41
3,0x743510ee9f21e24071c4e28edab4653df44ea620,351,6334,20060538.76,618751.25,1136815.99
4,0x9708fd9556116218fd29c778e9d81d255b61925b,177,6294,9432955.60,608337.64,766847.15


## 5 · Market-level volume summary

In [13]:
# join market metadata (question, end_date) for display
grouped_meta = grouped.merge(
    market_meta[["condition_id", "question", "end_date", "market_slug"]],
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 252,963


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3...,US government shutdown Saturday?,2026-01-31 00:00:00+00:00,5115,246651,129872244.40,133456584.59
1,0x655e5ca101c466b6293aa15e06173b78b293221803d5...,Will Zelenskyy wear a suit before July?,2025-06-30 00:00:00+00:00,2058,70116,129730833.42,130224636.85
2,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3...,Khamenei out as Supreme Leader of Iran by Febr...,2026-02-28 00:00:00+00:00,3975,154541,91963111.69,99206780.33
3,0xef8cf8b45ef7e3a607a72b6e1d56bede869fdd81795b...,TikTok banned in the US before May 2025?,2025-04-30 00:00:00+00:00,1260,38917,72454252.90,73173447.84
4,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,"US strikes Iran by February 28, 2026?",2026-01-31 00:00:00+00:00,4666,177785,63089422.45,68740787.56
5,0x32707ae194389a70f25314ec934851daba12d893da61...,"Will Eleven die in ""Stranger Things: Season 5""?",2025-12-31 00:00:00+00:00,1611,33628,60939071.45,60913170.56
6,0x77b4a1d4cd398a86c18b6bb9b5218917dc9f04b01a3c...,Will Polymarket US go live in 2025?,2025-12-31 00:00:00+00:00,3157,79846,46562345.16,47035404.25
7,0x62b0cd598091a179147acbd4616400f804acfdff6f76...,US x Venezuela military engagement by December...,2025-12-31 00:00:00+00:00,3052,107621,45077601.70,44399009.59
8,0x8ee2f1640386310eb5e7ffa596ba9335f2d324e303d2...,Russia x Ukraine ceasefire in 2025?,2025-12-31 00:00:00+00:00,4513,140915,44422109.49,42871214.77
9,0xf2ce8d3897ac5009a131637d3575f1f91c579bd08eec...,Xi Jinping out in 2025?,2025-12-31 00:00:00+00:00,2141,80428,41258645.74,42125953.41


## 6 · Wallet statistics (quantiles)

In [14]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["copyable_pnl"]    = wallet_stats["copyable_pnl"].round(2)
wallet_stats["trade_pnl"]    = wallet_stats["trade_pnl"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,copyable_pnl,trade_pnl,wallet_count_complement
quantile,,,,,,
0.00,80,1.00,1.00,-221026.98,-264410.50,80191
0.01,803,1.00,1.00,-22386.17,-29933.67,79468
0.05,4014,3.00,1.00,-3477.51,-5137.03,76257
0.10,8027,7.00,2.00,-1347.44,-2135.87,72244
0.25,20068,20.00,6.00,-247.05,-492.48,60203
0.50,40136,43.00,15.00,112.70,99.79,40135
0.75,60203,185.00,42.00,369.86,652.77,20068
0.90,72244,745.00,139.00,1057.50,2451.22,8027
0.95,76257,1806.00,290.00,2156.96,5518.27,4014


## 7 · Full enriched trade table

In [15]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,num_fills
0,0x7c0fecf5fb55d26ae8e855c2c1910832ed0aa552ca69...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-22 12:31:11+00:00,0x9c4b9abda44275da5a1f1b2822dd0d49f4adcfffb731...,Clippers,683.45,683.45,0.53,True,1.00,362.23,683.45,321.22,321.22,1
1,0x8e3ac2eeb09a1db32f2839685a174c97cc28ea05f18a...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-23 08:50:15+00:00,0xaaca3c792f030c897d98ea2f8a824d1842a3f31d054b...,Heat,1366.90,1366.90,0.50,True,1.00,683.45,1366.90,683.45,395.61,1
2,0xff336f7d5ee78018842d2ed0654df802cf40d725c8a1...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-24 12:45:17+00:00,0x1c1ece687825a5be15c65896cd0ee0f8cf06e1bc2a87...,Cavaliers,922.04,922.04,0.54,False,0.00,497.90,0.00,-497.90,-466.15,3
3,0x16d07796a1425eed307a85ea9d64411808115b84aa62...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-26 11:52:43+00:00,0x42b616c90a02e4e01bfed0aa673a1320f966e3381bef...,Hornets,1700.00,1700.00,0.30,False,0.00,510.00,0.00,-510.00,-20.01,8
4,0x468032632d1a852711e7e1232be55e2f0ce1ef885751...,0x00004a6d083e7d7f74a0b8b0e7fcda956193db52,BUY,2025-11-27 12:58:45+00:00,0xe77c389d59082205bc4f8881a21902aaecbde16521f6...,Packers,872.39,872.39,0.41,True,1.00,357.68,872.39,514.71,514.71,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57737367,0xef012966381427a28c11fa18dae0014cf1d2a8a7f59d...,0xffff50f0ac08761684731603b97ae806566235be,BUY,2026-02-17 21:15:07+00:00,0x7951237da1d15cc1335abc5c497098d4f324eae62b16...,Evansville Aces,542.00,542.00,0.80,False,0.00,433.60,0.00,-433.60,-268.80,2
57737368,0x522600758bf61fdc458a580a5b4a3aa306264d949b24...,0xffff50f0ac08761684731603b97ae806566235be,BUY,2026-03-06 12:27:35+00:00,0x61f6a8a39c42bc6711c7392862fb7e83a0235ded072a...,No,329.00,329.00,0.46,True,1.00,151.34,329.00,177.66,0.00,2
57737369,0xa810d5f07a792335cc86e5743e29705ff7f909603a4a...,0xffff50f0ac08761684731603b97ae806566235be,BUY,2026-03-16 20:48:29+00:00,0x667b6e62351dc8f54f40ec918ae4256dcca35b30bae8...,No,368.00,368.00,0.28,False,0.00,103.04,0.00,-103.04,0.00,1
57737370,0x931f472111b3bac3452021d3f27a6deeb363c6f34f5e...,0xffff50f0ac08761684731603b97ae806566235be,BUY,2026-03-17 13:02:33+00:00,0xc703fe394626096f832c3a269d47a46772e042c26827...,No,313.00,313.00,0.67,False,0.00,209.71,0.00,-209.71,0.00,1


## 8 · Export: apply PnL cut-off and write partitioned parquet

Filter grouped trades to ``final_wallets`` (wallets above the median per-shard PnL),
then write one parquet shard per first hex character of ``condition_id`` after ``0x``.

In [16]:
top_wallets = final_wallets
print(f"Wallets selected for export: {len(top_wallets):,}")

Wallets selected for export: 24,500


In [17]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl",
    "token_winner", "final_price",
    "tx_hash", "num_fills",
    "is_train","copyable_qty", "avail_copy_total_vol", "avail_copy_count",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_df = grouped[grouped["wallet"].isin(top_wallets)].copy()
export_df["is_train"] = export_df["dt"].dt.date <= END_DATE_TRAIN
export_df = export_df[EXPORT_COLS].reset_index(drop=True)

# Partition by the first hex character of condition_id after the "0x" prefix,
# matching the input shard layout (0.parquet … 9.parquet, a.parquet … f.parquet).
export_df["_shard"] = export_df["condition_id"].str[2]

output_files = []
for shard_key, shard_df in export_df.groupby("_shard", sort=True):
    out_file = OUT_DIR / f"{shard_key}.parquet"
    shard_df.drop(columns=["_shard"]).to_parquet(out_file, index=False)
    output_files.append(out_file)

export_df = export_df.drop(columns=["_shard"])
top_trades_preview   = export_df.head(5).copy()
top_trades_count     = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows exported:  {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
for f in sorted(output_files):
    print(f"  {f.name}  ({pd.read_parquet(f).shape[0]:,} rows)")
print(f"\nSaved partitioned output → {OUT_DIR}")

Total grouped rows exported:  9,787,589
  training rows: 7,111,315
  test rows:     2,676,274
Output shards:  16
  0.parquet  (586,370 rows)
  1.parquet  (573,866 rows)
  2.parquet  (614,811 rows)
  3.parquet  (623,212 rows)
  4.parquet  (660,898 rows)
  5.parquet  (599,250 rows)
  6.parquet  (657,866 rows)
  7.parquet  (586,816 rows)
  8.parquet  (608,427 rows)
  9.parquet  (586,399 rows)
  a.parquet  (693,882 rows)
  b.parquet  (593,362 rows)
  c.parquet  (556,485 rows)
  d.parquet  (633,485 rows)
  e.parquet  (584,543 rows)
  f.parquet  (627,917 rows)

Saved partitioned output → ../../data/polygon_trades_processed


In [18]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview

In [19]:
pd.read_parquet('../../data/polygon_trades_processed/0.parquet')

,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count
0,0x00046b741e19e9296863121475df84cc656b3746,0x0722102b758fff894d63c9d44ed47f24c99ca0486d746c82a5d06659c2e7d116,2026-01-09 08:26:42+00:00,BUY,Clippers,24.00,24.00,0.59,14.16,24.00,9.84,9.84,True,1.00,0x1c87b8cc6143ae2952e55266cefb1e7956a802b89bb0ff4aa2bf0308dc4e5e05,1,True,24.00,1478.88,15.00
1,0x00085583a8bed6e0fef83387c0f9a3753f00efa9,0x01588c543a49648dcb8016f0c7a6af1f1350e7b0bfdd343038ae47c6aa8ce151,2026-03-07 17:14:03+00:00,BUY,Heroic,610.00,610.00,0.17,103.70,0.00,-103.70,-103.70,False,0.00,0x1af9475d8d802a95778b6662c1267ef4224e482ceda2056afdf7c58cf4756fe1,1,False,610.00,1138.41,62.00
2,0x00085583a8bed6e0fef83387c0f9a3753f00efa9,0x01588c543a49648dcb8016f0c7a6af1f1350e7b0bfdd343038ae47c6aa8ce151,2026-03-07 17:16:33+00:00,BUY,Heroic,690.00,80.00,0.15,12.00,0.00,-12.00,-6.79,False,0.00,0xca394d19cfc6126959fd6f58b33e27e437eb5917fe5dae0bfa95ea6380086ae1,1,False,45.26,5.88,1.00
3,0x00085583a8bed6e0fef83387c0f9a3753f00efa9,0x01588c543a49648dcb8016f0c7a6af1f1350e7b0bfdd343038ae47c6aa8ce151,2026-03-07 17:25:53+00:00,BUY,Heroic,1020.00,330.00,0.15,49.50,0.00,-49.50,-40.54,False,0.00,0x69725cb73e12f9ddc21be2c2423d3d5f1b7e89219b3938d42592f7165cadaf1e,1,False,270.29,37.56,8.00
4,0x00085583a8bed6e0fef83387c0f9a3753f00efa9,0x01588c543a49648dcb8016f0c7a6af1f1350e7b0bfdd343038ae47c6aa8ce151,2026-03-07 18:18:45+00:00,BUY,Heroic,1053.18,33.18,0.15,4.98,0.00,-4.98,-4.98,False,0.00,0x0de7af89460eaeefab93d0c48d5bc42aec07e6d87981efeb887417c2230966cc,1,False,33.18,263.50,52.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586365,0xfff23ed024429dd3478bc2ac9c01f084a16a3f58,0x063f110f08e8a22d806225c2e9b6ffc6539f09b21e31b16212d5d6ce9d62d7bd,2026-02-17 15:00:47+00:00,SELL,Sakamoto,10.50,2.50,0.44,1.10,0.00,1.10,0.00,False,0.00,0x692f5f4f0c6bd7cde15e951296009860803298f26a56909522f3d3547cd050b8,1,False,-0.00,0.00,0.00
586366,0xfff23ed024429dd3478bc2ac9c01f084a16a3f58,0x063f110f08e8a22d806225c2e9b6ffc6539f09b21e31b16212d5d6ce9d62d7bd,2026-02-17 15:15:51+00:00,SELL,Sakamoto,4.50,6.00,0.44,2.64,0.00,2.64,2.64,False,0.00,0xaee31528951b9b923ecd84c68f578d76912246523cebdd0b064a1d6bce5f9069,1,False,6.00,4.60,1.00
586367,0xfff23ed024429dd3478bc2ac9c01f084a16a3f58,0x063f110f08e8a22d806225c2e9b6ffc6539f09b21e31b16212d5d6ce9d62d7bd,2026-02-17 15:15:53+00:00,SELL,Sakamoto,0.00,4.50,0.44,1.98,0.00,1.98,1.98,False,0.00,0x6e89fc8b0caa0e6978c13dd46a00a2cfec9a8e491e095ce6debe0a2b2423bfc8,1,False,4.50,4.60,1.00
586368,0xfffed6bab4cd49574152a8d77ab34086bc241df7,0x012f5743d45e2783931e0a4da18086874cd81c4a0fafda04454751a174829e9d,2025-06-05 18:49:52+00:00,BUY,Phillies,94.34,94.34,0.53,50.00,0.00,-50.00,0.00,False,0.00,0xa434f63d157b0f9ac0749209f3726715fbb0f8a344eb64e9f55cc430706a9e18,2,True,-0.00,0.00,0.00


### Unit test: CTF Exchange contract is not treated as a trader

`0x85d355ef32d62b09d2362184f299a7fc662ee1a4` is the Polymarket CTF Exchange
(on-chain matching contract). It can appear in per-side trade rows as a `wallet`
counterparty, but it is **not** a real trader.

This test always enforces that the exchange address is **not** in `top_wallets`.

In [20]:
CTF_EXCHANGE = "0x85d355ef32d62b09d2362184f299a7fc662ee1a4"

exchange_rows = grouped[grouped["wallet"] == CTF_EXCHANGE]
if len(exchange_rows) > 0:
    print(f"INFO  CTF Exchange appears in grouped:  {len(exchange_rows):,} rows")
else:
    print("INFO  CTF Exchange rows not present in current filtered dataset")

assert CTF_EXCHANGE not in top_wallets, (
    f"CTF Exchange contract ({CTF_EXCHANGE}) was incorrectly selected as a top wallet. "
    "It is the on-chain matching engine, not a real trader."
)

print(f"PASS  CTF Exchange not in top_wallets: top_wallets has {len(top_wallets):,} entries")

INFO  CTF Exchange rows not present in current filtered dataset
PASS  CTF Exchange not in top_wallets: top_wallets has 24,500 entries
